# CLDK, and an MCP server over it

CLDK is a Python SDK that runs static program analysis on Java, Python and TypeScript code and gives agents one API over them (Krishna et al., 2025). This notebook attaches it to the OWASP BenchmarkPython checkout HW1 used, asks the four questions an alert classifier needs, then serves those questions as MCP tools.

In [ ]:
import importlib.metadata, os, sys, subprocess
os.environ["SYSTEM_PYTHON"] = subprocess.run(["uv", "python", "find", "3.12"], capture_output=True, text=True).stdout.strip()
print(sys.executable); print("cldk", importlib.metadata.version("cldk")); print("mcp", importlib.metadata.version("mcp"))

## 1. Attach

The analysis object, and what the level means: symbol table, call graph, program dependency graph, system dependency graph. The first attach analyses 3,710 callables and takes minutes; later runs read the cache in `cldk-demo/owasp/.codeanalyzer`.

In [ ]:
import time
from cldk import CLDK
t = time.time()
an = CLDK.python(project_path="cldk-demo/owasp", analysis_level="program_dependency_graph")
print(len(an.get_callables_overview()), "callables,", f"{time.time()-t:.0f} s")

## 2. The four moves on a true positive

Bandit flags B608 at line 46 of `BenchmarkTest00099.py`. Move 1: which callable holds the line.

In [ ]:
hit = an.locate("testcode/BenchmarkTest00099.py", 46)
print(hit.callable.signature); print(hit.source[:400])

Move 2: the body, from the signature.

In [ ]:
print(an.get_method_bodies([hit.callable.signature])[hit.callable.signature])

Move 3: who calls it and what it calls.

In [ ]:
for c in an.callees_of(hit.callable.name): print(c.file, c.line, c.callable)
print(len(an.callers_of("get_safe_value")), "callers of the helper HW1's false positives share")

Move 4: does the request value reach the SQL string? The data-dependence graph of the handler, read by line. `resolve_value` and `flows_to_argument` resolve parameters, globals and captures, not local variables, so for a local the dependable call is `get_ddg`.

In [ ]:
import networkx as nx
def reaches(handler, src_line, dst_line):
    g = nx.DiGraph()
    for e in an.get_ddg(handler, page_size=500).edges:
        g.add_edge(e.src.split("@")[-1].split(":")[0], e.dst.split("@")[-1].split(":")[0], var=e.var)
    s, d = str(src_line), str(dst_line)
    if s in g and d in g and nx.has_path(g, s, d):
        p = nx.shortest_path(g, s, d); return p, [g[a][b]["var"] for a, b in zip(p, p[1:])]
    return None, None
print("00099: request (31) -> sql string (46):", reaches("BenchmarkTest00099_post", 31, 46))
print("00099: request (31) -> execute (49):   ", reaches("BenchmarkTest00099_post", 31, 49))

## 3. The same moves on a false positive

`BenchmarkTest00011.py`, B608 at line 47, where the f-string has no interpolation and the value goes into the parameter tuple.

In [ ]:
print("00011: request (39) -> sql string (47):", reaches("BenchmarkTest00011_post", 39, 47))
print("00011: request (39) -> execute (50):   ", reaches("BenchmarkTest00011_post", 39, 50))

Expected: a path for 00099 to both lines; for 00011 no path to the string and a path to the call through `bar`. That is the classifier's question answered as a path rather than a guess.

## 4. What an empty answer means

The agent API's rule: an empty result that could mean two things is a defect. On this checkout the entry-point report finds thousands of routes and marks almost all `app.route` decorators unresolved; read the coverage before trusting the list.

In [ ]:
cov = an.get_entrypoint_coverage(); eps = an.get_entrypoints()
print(len(eps), "entry points;", cov)

## 5. The server

Four tools over the same object, in the shape of lecture 5's server: `MCPServer`, `@mcp.tool`, `ToolError` for a path outside the checkout or a line in no callable.

In [ ]:
%%writefile cldk-demo/server.py
import os, subprocess
from pathlib import Path
from typing import TypedDict
import networkx as nx
from mcp.server import MCPServer                                # the SDK's main server class
from mcp.server.mcpserver.exceptions import ToolError           # raised when a tool refuses
from cldk import CLDK

HERE = Path(__file__).resolve().parent
TARGET = HERE / "owasp"
os.environ.setdefault("SYSTEM_PYTHON", subprocess.run(["uv", "python", "find", "3.12"], capture_output=True, text=True).stdout.strip())
AN = CLDK.python(project_path=str(TARGET), analysis_level="program_dependency_graph")
mcp = MCPServer("cldk")

def checked(path: str) -> str:
    p = (TARGET / path).resolve()
    if not p.is_relative_to(TARGET) or not p.is_file():
        raise ToolError(f"refused: {path} is not a file of the target")
    return p.relative_to(TARGET).as_posix()

class Located(TypedDict):
    signature: str | None
    name: str | None
    module: str

@mcp.tool(description="The callable that holds a line, or module scope.")
def locate(path: str, line: int) -> Located:
    hit = AN.locate(checked(path), line)
    return {"signature": hit.callable.signature if hit.callable else None,
            "name": hit.callable.name if hit.callable else None, "module": hit.module.module_name}

@mcp.tool(description="The source text of a callable, by signature.")
def source(signature: str) -> str:
    bodies = AN.get_method_bodies([signature])
    if signature not in bodies:
        raise ToolError(f"refused: no body for {signature}")
    return bodies[signature]

@mcp.tool(description="Callers of a callable by name, with file and line.")
def callers(name: str) -> list[dict]:
    return [{"file": c.file, "line": c.line, "callable": c.callable} for c in AN.callers_of(name)]

class Edge(TypedDict):
    from_line: int
    to_line: int
    var: str

@mcp.tool(description="Data-dependence edges of the callable that holds a line: from-line, to-line, variable.")
def dataflow(path: str, line: int) -> list[Edge]:
    hit = AN.locate(checked(path), line)
    if not hit.callable:
        raise ToolError(f"refused: line {line} of {path} is in no callable")
    page = AN.get_ddg(hit.callable.signature, page_size=2000)  # full signature: a bare name like init matches 1,230 callables
    out = []
    for e in page.edges:
        try:
            out.append({"from_line": int(e.src.split("@")[-1].split(":")[0]), "to_line": int(e.dst.split("@")[-1].split(":")[0]), "var": e.var})
        except ValueError:
            continue
    return out

if __name__ == "__main__":
    mcp.run()

## 6. The Inspector

The catalog and the command, as in lecture 5.

In [ ]:
import json, pathlib
cat = {"mcpServers": {"cldk": {"type": "stdio", "command": str(pathlib.Path(".venv/bin/python").resolve()), "args": [str(pathlib.Path("cldk-demo/server.py").resolve())]}}}
pathlib.Path("cldk-demo/catalog.json").write_text(json.dumps(cat, indent=2)); print(json.dumps(cat, indent=2))

In a second terminal: `npx @modelcontextprotocol/inspector --web --catalog cldk-demo/catalog.json`. Run section 1 first. The server attaches to the checkout when it starts, from the cache that section 1 builds; a start without that cache takes minutes, and the Inspector gives up on a server that takes longer than 30 seconds to answer. In class: `locate` on `testcode/BenchmarkTest00099.py` line 46, `dataflow` on the same, then `locate` on `../expectedresults-0.1.csv` to see the refusal.